In [22]:
import pandas as pd
import numpy as np
import json
import warnings
from pathlib import Path

warnings.filterwarnings("ignore")

# ─── Пути ────────────────────────────────────────────────────────────────────
DATA_DIR   = Path("data_sensors")   # Сюда кидаешь Excel файлы
CONFIG_DIR = Path("configs")
CONFIG_DIR.mkdir(parents=True, exist_ok=True)

# ─── Параметры очистки ────────────────────────────────────────────────────────
MAX_CAPACITY  = 150   # Физический максимум трёхсекционного трамвая
MIN_DAYS      = 3     # Минимум дней наблюдений для включения пары (остановка, час)
ASSUMED_SPEED = 15.0  # км/ч для оценки расстояний по времени хода

# ─── Маршруты ─────────────────────────────────────────────────────────────────
ROUTES     = ["20", "48", "55"]
DIRECTIONS = ["Прямое", "Обратное"]


In [23]:
def load_all_data(data_path: Path) -> pd.DataFrame:
    all_files = list(data_path.glob("*.xlsx"))
    if not all_files:
        raise FileNotFoundError(f"Excel файлы не найдены в {data_path.resolve()}")

    # Жёстко задаём имена: двойная шапка пропускается через skiprows=2
    COLUMNS = [
        "date", "route", "direction", "board_num", "trip_id",
        "stop_seq", "_stop_id", "stop_name", "arr_time",
        "boarded", "alighted", "in_cabin", "veh_type"
    ]

    df_list = []
    for f in all_files:
        df = pd.read_excel(f, skiprows=2, names=COLUMNS)
        df["_source_file"] = f.name   # для отладки
        df_list.append(df)

    df_raw = pd.concat(df_list, ignore_index=True)

    # Сразу выкидываем колонки, которые нам не нужны
    df_raw = df_raw.drop(columns=["_stop_id", "veh_type"])

    print(f"Загружено файлов: {len(all_files)}")
    print(f"Всего строк:      {len(df_raw):,}")
    return df_raw


df_raw = load_all_data(DATA_DIR)


Загружено файлов: 24
Всего строк:      331,316


In [24]:
def clean_data(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    n0 = len(df)

    # 1. Принудительно конвертируем числа — строки/NaN становятся NaN и дропаются
    for col in ["boarded", "alighted", "in_cabin", "stop_seq"]:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # 2. Время
    df["arr_time"] = pd.to_datetime(df["arr_time"], errors="coerce")

    # 3. Нормализуем строковые поля
    for col in ["route", "direction", "stop_name", "trip_id", "board_num"]:
        df[col] = df[col].astype(str).str.strip()

    # 4. Удаляем строки с критическими пропусками
    df = df.dropna(subset=["arr_time", "route", "direction", "trip_id", "stop_seq"])
    print(f"  После dropna:               {len(df):,} ({len(df)/n0*100:.1f}%)")

    # 5. Отсечка физически невозможных значений ─ включая in_cabin!
    for col in ["boarded", "alighted", "in_cabin"]:
        df = df[df[col].between(0, MAX_CAPACITY)]
    print(f"  После отсечки >{MAX_CAPACITY}:          {len(df):,} ({len(df)/n0*100:.1f}%)")

    # 6. Удаляем полностью "мёртвые" рейсы
    #    Рейс мёртв, если во ВСЕХ строках boarded + alighted + in_cabin == 0
    trip_sum = df.groupby("trip_id")[["boarded", "alighted", "in_cabin"]].sum()
    dead_trips = trip_sum[
        (trip_sum["boarded"] == 0) &
        (trip_sum["alighted"] == 0) &
        (trip_sum["in_cabin"] == 0)
    ].index
    df = df[~df["trip_id"].isin(dead_trips)]
    print(f"  После удаления мёртвых рейсов: {len(df):,} ({len(df)/n0*100:.1f}%)")

    # 7. Дополнительные колонки для агрегации
    df["hour"]      = df["arr_time"].dt.hour
    df["date_only"] = df["arr_time"].dt.date

    return df.reset_index(drop=True)


print("=== Очистка данных ===")
df_clean = clean_data(df_raw)

# Быстрая проверка — убеждаемся, что in_cabin > 150 не осталось
assert df_clean["in_cabin"].max() <= MAX_CAPACITY, "in_cabin всё ещё содержит выбросы!"
assert df_clean["boarded"].max() <= MAX_CAPACITY,  "boarded всё ещё содержит выбросы!"
print(f"\n  in_cabin max: {df_clean['in_cabin'].max()}")
print(f"  boarded  max: {df_clean['boarded'].max()}")


=== Очистка данных ===
  После dropna:               284,379 (85.8%)
  После отсечки >150:          281,237 (84.9%)
  После удаления мёртвых рейсов: 80,762 (24.4%)

  in_cabin max: 150
  boarded  max: 130


In [25]:
def build_reference_route(df: pd.DataFrame, route: str, direction: str) -> pd.DataFrame:
    """
    Строит эталонную последовательность остановок по stop_seq.
    Привязка идёт по порядковому номеру, stop_name только для подписей.
    """
    d = df[(df["route"] == route) & (df["direction"] == direction)]

    if d.empty:
        print(f"  ПРЕДУПРЕЖДЕНИЕ: нет данных для маршрута {route} ({direction})")
        return pd.DataFrame(columns=["stop_seq", "stop_name", "clean_stop_id"])

    # Для каждого stop_seq берём самое частое название
    ref = (d.groupby("stop_seq")["stop_name"]
            .agg(lambda x: x.mode().iloc[0])
            .reset_index()
            .sort_values("stop_seq")
            .reset_index(drop=True))

    # clean_stop_id — непрерывная нумерация с 1 без дырок
    ref["clean_stop_id"] = np.arange(1, len(ref) + 1, dtype=int)

    print(f"  Маршрут {route} ({direction}): {len(ref)} остановок "
          f"(seq {int(ref['stop_seq'].min())}–{int(ref['stop_seq'].max())})")
    return ref


def attach_clean_stop_id(df: pd.DataFrame, ref: pd.DataFrame) -> pd.DataFrame:
    """Добавляет clean_stop_id к строкам через словарь stop_seq → clean_stop_id."""
    mapping = dict(zip(ref["stop_seq"], ref["clean_stop_id"]))
    out = df.copy()
    out["clean_stop_id"] = out["stop_seq"].map(mapping)
    return out.dropna(subset=["clean_stop_id"]).assign(
        clean_stop_id=lambda x: x["clean_stop_id"].astype(int)
    )


# Строим эталоны для всех маршрутов и направлений
print("=== Эталонные маршруты ===")
ref_routes = {}
for route in ROUTES:
    for direction in DIRECTIONS:
        key = (route, direction)
        ref_routes[key] = build_reference_route(df_clean, route, direction)


=== Эталонные маршруты ===
  Маршрут 20 (Прямое): 34 остановок (seq 0–33)
  Маршрут 20 (Обратное): 33 остановок (seq 0–32)
  Маршрут 48 (Прямое): 36 остановок (seq 0–35)
  Маршрут 48 (Обратное): 35 остановок (seq 0–34)
  Маршрут 55 (Прямое): 38 остановок (seq 1–38)
  Маршрут 55 (Обратное): 37 остановок (seq 2–38)


In [26]:
def estimate_distances(df: pd.DataFrame, ref: pd.DataFrame) -> list:
    """
    Вычисляет расстояния через медианное время хода между соседними остановками.
    time_diff считается внутри каждого рейса (сортировка по trip_id + arr_time).
    """
    if ref.empty:
        return []

    d = attach_clean_stop_id(df, ref)

    # Сортируем по рейсу и времени прибытия
    d = d.sort_values(["trip_id", "arr_time"])

    # Время хода до текущей остановки внутри рейса (в минутах)
    d["time_diff_min"] = (d.groupby("trip_id")["arr_time"]
                           .diff()
                           .dt.total_seconds() / 60.0)

    n = len(ref)
    distances = [[1, 0]]  # Первая остановка — начало маршрута

    for stop_id in range(2, n + 1):
        stop_data = d[d["clean_stop_id"] == stop_id]["time_diff_min"]

        # Фильтруем аномалии: 0.5 — 15 мин (трамвай не едет мгновенно и не стоит вечно)
        valid = stop_data[(stop_data >= 0.5) & (stop_data <= 15.0)]

        if not valid.empty:
            median_min = valid.median()
        else:
            median_min = 2.0  # Дефолт — 2 минуты

        # Дистанция в метрах, округлённая до 50 м, минимум 100 м
        dist_m = (median_min / 60.0) * ASSUMED_SPEED * 1000
        dist_m = max(100, int((dist_m // 50) * 50))
        distances.append([stop_id, dist_m])

    return distances


In [27]:
def calculate_intensity(df: pd.DataFrame, ref: pd.DataFrame) -> list:
    """
    Считает среднее число посадок на каждой остановке в каждый час.
    Делим на количество дней, В КОТОРЫХ ЕСТЬ НАБЛЮДЕНИЯ.
    """
    if ref.empty:
        return []

    d = attach_clean_stop_id(df, ref)
    g = d.groupby(["clean_stop_id", "hour", "date_only"])["boarded"].sum().reset_index()
    g_sum  = g.groupby(["clean_stop_id", "hour"])["boarded"].sum()
    g_days = g.groupby(["clean_stop_id", "hour"])["date_only"].nunique()

    mask = g_days >= MIN_DAYS
    avg = (g_sum[mask] / g_days[mask]).fillna(0).round().astype(int).reset_index()
    avg.columns = ["clean_stop_id", "hour", "intensity"]

    result = [
        [int(r.clean_stop_id), int(r.hour), int(r.intensity)]
        for r in avg.itertuples(index=False)
    ]
    return sorted(result, key=lambda x: (x[0], x[1]))


def fill_intensity_gaps(intensity: list, stop_number: int) -> list:
    """
    Заполняет "дырки" в часах для каждой остановки:
    - Внутри дня (между известными значениями) -> линейная интерполяция
    - Края дня (утро/вечер) -> затухание до нуля
    """
    if not intensity:
        return []

    # Группируем данные по остановкам: {stop_id: {hour: value}}
    data_dict = {i: {} for i in range(1, stop_number + 1)}
    for stop_id, hour, val in intensity:
        data_dict[stop_id][hour] = val

    filled_intensity = []
    
    for stop_id in range(1, stop_number + 1):
        hours_data = data_dict[stop_id]
        
        # Если для остановки вообще нет данных - заполняем нулями
        if not hours_data:
            for h in range(24):
                filled_intensity.append([stop_id, h, 0])
            continue
            
        known_hours = sorted(hours_data.keys())
        first_h = known_hours[0]
        last_h = known_hours[-1]
        
        for h in range(24):
            if h in hours_data:
                # Значение известно
                val = hours_data[h]
            elif first_h < h < last_h:
                # Дырка ВНУТРИ дня -> линейная интерполяция
                # Ищем ближайшего соседа слева и справа
                left_h = max([kh for kh in known_hours if kh < h])
                right_h = min([kh for kh in known_hours if kh > h])
                
                left_v = hours_data[left_h]
                right_v = hours_data[right_h]
                
                # Пропорция между левым и правым значением
                ratio = (h - left_h) / (right_h - left_h)
                val = int(round(left_v + (right_v - left_v) * ratio))
            elif h < first_h:
                # Дырка УТРОМ -> плавное нарастание от нуля к первому значению
                if first_h >= 6: # Если первое значение хотя бы в 6 утра
                    ratio = (h - 4) / (first_h - 4) if h >= 4 else 0 # Начинаем рост с 4 утра
                    ratio = max(0, min(1, ratio))
                    val = int(round(hours_data[first_h] * ratio))
                else:
                    val = 0
            else: # h > last_h
                # Дырка ВЕЧЕРОМ -> плавное затухание от последнего значения к нулю
                if last_h <= 22:
                    ratio = (23 - h) / (23 - last_h) if h <= 23 else 0
                    ratio = max(0, min(1, ratio))
                    val = int(round(hours_data[last_h] * ratio))
                else:
                    val = 0
                    
            filled_intensity.append([stop_id, h, max(0, val)])

    return sorted(filled_intensity, key=lambda x: (x[0], x[1]))


def intensity_health(intensity: list, label: str = ""):
    total = len(intensity)
    nonzero = sum(v[2] > 0 for v in intensity)
    pct = (nonzero / total * 100) if total > 0 else 0
    print(f"  {label:30s} intensity entries: {total:4d}, "
          f"non-zero: {nonzero:4d} ({pct:.1f}%)")


In [28]:
def calculate_intervals(df: pd.DataFrame, ref: pd.DataFrame) -> list:
    """
    Считает средний интервал между трамваями по часам.
    Логика: считаем уникальные trip_id в каждый (день, час),
    усредняем по дням и переводим в интервал: 60 / avg_trips_per_hour.
    """
    if ref.empty:
        return [[6, 15]]  # Дефолт если данных нет

    d = attach_clean_stop_id(df, ref)

    trips_per_hour = (d.groupby(["date_only", "hour"])["trip_id"]
                       .nunique()
                       .reset_index()
                       .rename(columns={"trip_id": "trip_count"}))

    avg_per_hour = trips_per_hour.groupby("hour")["trip_count"].mean()

    intervals = []
    for hour in range(24):
        if hour in avg_per_hour.index and avg_per_hour[hour] > 0:
            interval = int(round(60.0 / avg_per_hour[hour]))
            interval = max(3, min(interval, 60))  # Зажимаем в [3, 60] минут
        else:
            # Нет данных → дефолтные значения по времени суток
            if 7 <= hour <= 9 or 17 <= hour <= 19:
                interval = 8    # Час пик
            elif 6 <= hour <= 22:
                interval = 15   # Обычное время
            else:
                interval = 30   # Ночь

        intervals.append([hour, interval])

    # Убираем дубли (если несколько часов подряд одинаковый интервал)
    compact = [intervals[0]]
    for hour, ivl in intervals[1:]:
        if ivl != compact[-1][1]:
            compact.append([hour, ivl])

    return compact


In [29]:
def save_config_compact(config: dict, output_file: Path):
    """
    Записывает конфиг в читаемом формате:
    - distance, bus_interval, road_loads — одной строкой
    - intensity — по одной строке на остановку (пустые пропускаются)
    - скалярные поля — по одному на строку
    """
    ARRAY_ONELINE = ["distance", "bus_interval", "road_loads"]
    SCALAR_ORDER  = [
        "flow_speed", "peak_stop", "tram_capacity", "tram_count",
        "operation_start_hour", "operation_end_hour", "simulation_hours",
        "acceleration_time", "stop_time", "turnaround_time",
        "target_utilization", "random_seed"
    ]

    lines = ["{"]
    lines.append(f'  "stop_number": {config["stop_number"]},')

    for key in ARRAY_ONELINE:
        if key in config:
            lines.append(f'  "{key}": {json.dumps(config[key], ensure_ascii=False)},')

    # intensity — по строке на остановку
    lines.append('  "intensity": [')
    n = config["stop_number"]
    
    # Сначала собираем только те остановки, где РЕАЛЬНО есть данные
    intensity_lines = []
    for stop_id in range(1, n + 1):
        items = [x for x in config["intensity"] if x[0] == stop_id]
        if items:
            # Склеиваем массивы для одной остановки в строку: [1, 0, 5], [1, 1, 10]
            s = ", ".join(json.dumps(x, ensure_ascii=False) for x in items)
            intensity_lines.append(s)
            
    # Записываем с правильными запятыми (последняя строка без запятой)
    for i, text_line in enumerate(intensity_lines):
        comma = "," if i < len(intensity_lines) - 1 else ""
        lines.append(f"    {text_line}{comma}")
        
    lines.append("  ],")

    # Скалярные поля в заданном порядке
    existing_scalars = [k for k in SCALAR_ORDER if k in config]
    for i, key in enumerate(existing_scalars):
        is_last = (i == len(existing_scalars) - 1)
        comma   = "" if is_last else ","
        lines.append(f'  "{key}": {json.dumps(config[key], ensure_ascii=False)}{comma}')

    lines.append("}")

    output_file.write_text("\n".join(lines), encoding="utf-8")
    print(f"  Сохранено: {output_file.name}")


In [30]:
def build_config_for_route(df: pd.DataFrame, route: str, direction: str) -> dict | None:
    key = (route, direction)
    ref = ref_routes.get(key)

    if ref is None or ref.empty:
        print(f"  Нет эталона для {route} ({direction}), пропускаем")
        return None

    d = df[(df["route"] == route) & (df["direction"] == direction)]
    stop_number = len(ref)

    distances     = estimate_distances(d, ref)
    raw_intensity = calculate_intensity(d, ref)
    intervals     = calculate_intervals(d, ref)
    
    # ВОТ ЗДЕСЬ ЗАПОЛНЯЕМ ДЫРКИ В ЧАСАХ!
    filled_intensity = fill_intensity_gaps(raw_intensity, stop_number)

    config = {
        "stop_number": stop_number,
        "distance": distances,
        "intensity": filled_intensity,  # Используем заполненную версию
        "bus_interval": intervals,
        "road_loads": [
            [0, 0.1], [6, 0.4], [7, 0.7], [8, 0.9],
            [10, 0.6], [16, 0.7], [17, 0.9], [20, 0.4], [23, 0.1]
        ],
        "flow_speed": int(ASSUMED_SPEED),
        "peak_stop": (stop_number + 1) // 2,
        "tram_capacity": MAX_CAPACITY,
        "tram_count": 8,
        "operation_start_hour": 6,
        "operation_end_hour": 24,
        "simulation_hours": 24,
        "acceleration_time": 0.5,
        "stop_time": 1.0,
        "turnaround_time": 2.0,
        "target_utilization": 0.75,
        "random_seed": 42
    }

    return config


print("=== Генерация конфигов ===\n")
all_configs = {}

for route in ROUTES:
    for direction in DIRECTIONS:
        print(f"Маршрут {route} ({direction}):")
        cfg = build_config_for_route(df_clean, route, direction)
        if cfg is None:
            continue

        intensity_health(cfg["intensity"], label=f"{route} {direction}")

        # Имя файла: route_20_fwd_config.json
        dir_tag = "fwd" if direction == "Прямое" else "bwd"
        out_path = CONFIG_DIR / f"route_{route}_{dir_tag}_config.json"
        
        save_config_compact(cfg, out_path)
        all_configs[(route, direction)] = cfg
        print()


=== Генерация конфигов ===

Маршрут 20 (Прямое):
  20 Прямое                      intensity entries:  816, non-zero:  606 (74.3%)
  Сохранено: route_20_fwd_config.json

Маршрут 20 (Обратное):
  20 Обратное                    intensity entries:  792, non-zero:  619 (78.2%)
  Сохранено: route_20_bwd_config.json

Маршрут 48 (Прямое):
  48 Прямое                      intensity entries:  864, non-zero:  492 (56.9%)
  Сохранено: route_48_fwd_config.json

Маршрут 48 (Обратное):
  48 Обратное                    intensity entries:  840, non-zero:  443 (52.7%)
  Сохранено: route_48_bwd_config.json

Маршрут 55 (Прямое):
  55 Прямое                      intensity entries:  912, non-zero:  657 (72.0%)
  Сохранено: route_55_fwd_config.json

Маршрут 55 (Обратное):
  55 Обратное                    intensity entries:  888, non-zero:  643 (72.4%)
  Сохранено: route_55_bwd_config.json



In [31]:
print("=" * 60)
print("ИТОГО")
print("=" * 60)

for (route, direction), cfg in all_configs.items():
    n_stops = cfg["stop_number"]
    n_int   = len(cfg["intensity"])
    nz      = sum(v[2] > 0 for v in cfg["intensity"])
    max_int = max((v[2] for v in cfg["intensity"]), default=0)
    avg_d   = np.mean([d[1] for d in cfg["distance"][1:]]) if n_stops > 1 else 0

    print(f"\n  {route} {direction:8s}:")
    print(f"    Остановок:              {n_stops}")
    print(f"    Ср. расстояние:         {avg_d:.0f} м")
    print(f"    Intensity entries:      {n_int}")
    print(f"    Non-zero intensity:     {nz} ({(nz/n_int*100 if n_int else 0):.1f}%)")
    print(f"    Макс. intensity/ч:      {max_int}")

print("\n" + "=" * 60)
print(f"Конфиги сохранены в: {CONFIG_DIR.resolve()}")
print("=" * 60)


ИТОГО

  20 Прямое  :
    Остановок:              34
    Ср. расстояние:         412 м
    Intensity entries:      816
    Non-zero intensity:     606 (74.3%)
    Макс. intensity/ч:      24

  20 Обратное:
    Остановок:              33
    Ср. расстояние:         414 м
    Intensity entries:      792
    Non-zero intensity:     619 (78.2%)
    Макс. intensity/ч:      29

  48 Прямое  :
    Остановок:              36
    Ср. расстояние:         419 м
    Intensity entries:      864
    Non-zero intensity:     492 (56.9%)
    Макс. intensity/ч:      14

  48 Обратное:
    Остановок:              35
    Ср. расстояние:         404 м
    Intensity entries:      840
    Non-zero intensity:     443 (52.7%)
    Макс. intensity/ч:      24

  55 Прямое  :
    Остановок:              38
    Ср. расстояние:         531 м
    Intensity entries:      912
    Non-zero intensity:     657 (72.0%)
    Макс. intensity/ч:      30

  55 Обратное:
    Остановок:              37
    Ср. расстояние:        